# 00 Master Run All

This is the notebook-first entry point for the complete audio project. It runs the pipeline in the correct order, with explicit gates before expensive stages.

Keep `RUN_CNN = False` until dataset audit, split, quality, features, and ML baselines look correct. Set it to true only on the GPU machine.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "covid_audio_btp").exists():
            return candidate
    raise FileNotFoundError("Could not find project root. Start Jupyter from the extracted covid_audio_btp folder or one of its subfolders.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from covid_audio_btp.notebook_utils import (
    PROJECT_ROOT,
    artifact,
    check_artifacts,
    count_table,
    read_csv,
    read_optional_csv,
    require_artifacts,
    save_table,
    value_counts_frame,
    assert_no_participant_leakage,
    assert_binary_labels_present,
    stop_if_validation_errors,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
print(PROJECT_ROOT)


## Configuration

Put Coswara under `data/raw/coswara` unless you intentionally use another local path. The notebook does not download private datasets.


In [ ]:
RAW_COSWARA_DIR = PROJECT_ROOT / "data/raw/coswara"
RUN_LAYOUT_AUDIT = True
RUN_CNN = False
CNN_EPOCHS = 50
CNN_BATCH_SIZE = 32

RAW_COSWARA_DIR


## Artifact Dashboard

This tells you what already exists before running anything. Missing artifacts are expected on the first run.


In [ ]:
CORE_ARTIFACTS = [
    "data/interim/coswara_index.csv",
    "data/processed/metadata_clean.csv",
    "data/interim/modality_availability.csv",
    "data/interim/split_manifest.csv",
    "data/processed/audio_quality.csv",
    "reports/tables/validation_issues.csv",
    "data/processed/features_mfcc.csv",
    "data/processed/spectrogram_index.csv",
    "data/outputs/metrics/ml_baseline_metrics.csv",
    "data/outputs/metrics/ml_predictions_validation.csv",
    "data/outputs/metrics/ml_predictions_test.csv",
    "data/outputs/metrics/calibration_metrics.csv",
    "data/outputs/metrics/calibrated_branch_predictions.csv",
    "data/outputs/metrics/fusion_metrics.csv",
    "data/outputs/metrics/subgroup_metrics.csv",
]
check_artifacts(CORE_ARTIFACTS)


## Stage 0: Environment And Raw Layout

The raw-layout audit is intentionally before indexing. It catches wrong dataset placement and unexpected Coswara directory structures early.


In [ ]:
def run_cmd(args):
    print("$", " ".join(map(str, args)))
    result = subprocess.run(args, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
    return result

run_cmd([sys.executable, "scripts/00_check_environment.py"])

if not RAW_COSWARA_DIR.exists():
    raise FileNotFoundError(
        f"Put Coswara at {RAW_COSWARA_DIR} or change RAW_COSWARA_DIR in this notebook."
    )

if RUN_LAYOUT_AUDIT:
    run_cmd([
        sys.executable,
        "scripts/00_inspect_dataset_layout.py",
        "--raw-dir", str(RAW_COSWARA_DIR),
        "--output", "reports/tables/coswara_layout_audit.csv",
    ])


## Stage 1: Index, Metadata, Splits, Quality

This must finish before any model work. It prevents the Week-6 regression loop from the old plan.


In [ ]:
run_cmd([
    sys.executable, "scripts/01_build_coswara_index.py",
    "--raw-dir", str(RAW_COSWARA_DIR),
    "--output", "data/interim/coswara_index.csv",
])
run_cmd([
    sys.executable, "scripts/02_clean_metadata.py",
    "--index", "data/interim/coswara_index.csv",
    "--output", "data/processed/metadata_clean.csv",
    "--availability-output", "data/interim/modality_availability.csv",
    "--audit-output", "reports/tables/dataset_audit.csv",
])
run_cmd([
    sys.executable, "scripts/03_create_splits.py",
    "--metadata", "data/processed/metadata_clean.csv",
    "--output", "data/interim/split_manifest.csv",
    "--metadata-output", "data/processed/metadata_clean.csv",
    "--audit-output", "reports/tables/split_audit.csv",
])
run_cmd([
    sys.executable, "scripts/04_quality_audit.py",
    "--metadata", "data/processed/metadata_clean.csv",
    "--output", "data/processed/audio_quality.csv",
    "--metadata-output", "data/processed/metadata_clean.csv",
    "--summary-output", "reports/tables/quality_summary.csv",
])
run_cmd([sys.executable, "scripts/12_validate_artifacts.py", "--strict"])


## Hard Gate: Dataset Validity

Stop here if labels are unknown, participant leakage exists, or quality coverage is too low. Do not train models before this passes.


In [ ]:
metadata = read_csv("data/processed/metadata_clean.csv", n=3)
quality = read_csv("data/processed/audio_quality.csv", n=3)
issues = read_optional_csv("reports/tables/validation_issues.csv")

assert_binary_labels_present(metadata)
assert_no_participant_leakage(metadata)
if issues is not None:
    stop_if_validation_errors(issues)

quality_ok_rate = (quality["quality_flag"] == "ok").mean() if len(quality) else 0.0
print(f"quality ok rate: {quality_ok_rate:.3f}")
if quality_ok_rate < 0.50:
    raise AssertionError("Quality ok rate is below 50%; inspect 02_quality_review.ipynb before continuing.")


## Stage 2: Features And Spectrograms

The feature script uses event-aware preprocessing before fixed-length feature/spectrogram formatting.


In [ ]:
run_cmd([
    sys.executable, "scripts/05_extract_features.py",
    "--metadata", "data/processed/metadata_clean.csv",
    "--features-output", "data/processed/features_mfcc.csv",
    "--spectrogram-dir", "data/processed/spectrograms",
    "--spectrogram-index-output", "data/processed/spectrogram_index.csv",
    "--quality-mode", "all_samples",
])
require_artifacts(["data/processed/features_mfcc.csv", "data/processed/spectrogram_index.csv"])


## Stage 3: Classical Baselines

These models establish whether the dataset and features are sane before CNN training.


In [ ]:
run_cmd([
    sys.executable, "scripts/06_train_ml_baselines.py",
    "--features", "data/processed/features_mfcc.csv",
    "--models-dir", "data/outputs/models",
    "--metrics-output", "data/outputs/metrics/ml_baseline_metrics.csv",
    "--validation-output", "data/outputs/metrics/ml_predictions_validation.csv",
    "--test-output", "data/outputs/metrics/ml_predictions_test.csv",
])
ml_metrics = read_csv("data/outputs/metrics/ml_baseline_metrics.csv")
ml_metrics.sort_values(["modality", "auprc"], ascending=[True, False]).head(20)


## Optional Stage 4: Compact CNN

Run only on the GPU machine after the classical baseline audit looks reasonable.


In [ ]:
if RUN_CNN:
    for modality in ["cough", "breath", "speech"]:
        run_cmd([
            sys.executable, "scripts/07_train_cnn.py",
            "--spectrogram-index", "data/processed/spectrogram_index.csv",
            "--modality", modality,
            "--models-dir", "data/outputs/models",
            "--metrics-output", f"data/outputs/metrics/cnn_metrics_{modality}.csv",
            "--validation-output", f"data/outputs/metrics/cnn_logits_validation_{modality}.csv",
            "--test-output", f"data/outputs/metrics/cnn_logits_test_{modality}.csv",
            "--history-output", f"data/outputs/metrics/cnn_training_history_{modality}.csv",
            "--epochs", str(CNN_EPOCHS),
            "--batch-size", str(CNN_BATCH_SIZE),
        ])
else:
    print("RUN_CNN is False. Keep it false until earlier audits pass and you are on GPU.")


## Stage 5: Calibration, Fusion, Subgroup Checks, Report Assets

Branch calibration happens before fusion, and fusion includes both uniform and validation-weighted baselines.


In [ ]:
run_cmd([
    sys.executable, "scripts/08_calibrate_branches.py",
    "--validation-predictions", "data/outputs/metrics/ml_predictions_validation.csv",
    "--test-predictions", "data/outputs/metrics/ml_predictions_test.csv",
    "--output", "data/outputs/metrics/calibrated_branch_predictions.csv",
    "--metrics-output", "data/outputs/metrics/calibration_metrics.csv",
    "--method", "platt",
])
run_cmd([
    sys.executable, "scripts/09_run_fusion.py",
    "--predictions", "data/outputs/metrics/calibrated_branch_predictions.csv",
    "--validation-metrics", "data/outputs/metrics/ml_validation_metrics.csv",
    "--output", "data/outputs/metrics/fusion_predictions.csv",
    "--metrics-output", "data/outputs/metrics/fusion_metrics.csv",
])
run_cmd([
    sys.executable, "scripts/10_shift_and_confounding_checks.py",
    "--predictions", "data/outputs/metrics/fusion_predictions.csv",
    "--metadata", "data/processed/metadata_clean.csv",
    "--output", "data/outputs/metrics/subgroup_metrics.csv",
])
run_cmd([
    sys.executable, "scripts/11_make_report_assets.py",
    "--metadata", "data/processed/metadata_clean.csv",
    "--predictions", "data/outputs/metrics/fusion_predictions.csv",
])
check_artifacts(CORE_ARTIFACTS)
